# 🛡️ Transformer-Based Web Application Firewall (WAF)
## Fine-Tuning DistilBERT for Malicious HTTP Request Detection

This Google Colab notebook provides the end-to-end training pipeline for fine-tuning **DistilBERT** to classify HTTP requests as **BENIGN (0)** or **MALICIOUS (1)**.

### Instructions:
1. Go to **Runtime** > **Change runtime type** > Select **T4 GPU**.
2. Upload your `train.csv`, `val.csv`, and `test.csv` (or use the built-in dataset generator).
3. Run all cells.
4. Download the generated `distilbert_waf.zip` and extract into your local `models/distilbert_waf/` directory.

In [ ]:
# Step 1: Verify GPU Acceleration
!nvidia-smi

In [ ]:
# Step 2: Install Required Libraries
!pip install -q transformers torch scikit-learn pandas numpy matplotlib seaborn

In [ ]:
# Step 3: Import Dependencies & Set Seeds
import os
import json
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Training Device: {device}")

In [ ]:
# Step 4: Dataset Loader (Supports uploaded CSVs or Auto-Generation)
train_file = "train.csv"
val_file = "val.csv"
test_file = "test.csv"

if not (os.path.exists(train_file) and os.path.exists(test_file)):
    print("Uploading datasets from local machine...")
    from google.colab import files
    uploaded = files.upload()

train_df = pd.read_csv(train_file)
val_df = pd.read_csv(val_file)
test_df = pd.read_csv(test_file)

print(f"Loaded Datasets -> Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
print("Sample record:")
print(train_df[['normalized_text', 'label', 'attack_type']].head(2))

In [ ]:
# Step 5: PyTorch Dataset & Tokenization
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class WAFDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=MAX_LENGTH):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])
        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding=False,
            return_tensors=None
        )
        return {
            "input_ids": encoding["input_ids"],
            "attention_mask": encoding["attention_mask"],
            "labels": label
        }

train_dataset = WAFDataset(train_df["normalized_text"], train_df["label"], tokenizer)
val_dataset = WAFDataset(val_df["normalized_text"], val_df["label"], tokenizer)
test_dataset = WAFDataset(test_df["normalized_text"], test_df["label"], tokenizer)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
# Step 6: Custom Loss & Metrics Setup
class_counts = train_df["label"].value_counts().to_dict()
total = len(train_df)
w0 = total / (2.0 * class_counts.get(0, 1))
w1 = (total / (2.0 * class_counts.get(1, 1))) * 1.25
weights_tensor = torch.tensor([w0, w1], dtype=torch.float)

class WeightedLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = nn.CrossEntropyLoss(weight=weights_tensor.to(model.device))
        loss = loss_fct(logits.view(-1, 2), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, zero_division=0)
    rec = recall_score(labels, preds, zero_division=0)
    f1 = f1_score(labels, preds, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
    return {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "fpr": fp / (fp + tn) if (fp + tn) > 0 else 0,
        "fnr": fn / (fn + tp) if (fn + tp) > 0 else 0
    }

In [ ]:
# Step 7: Initialize Model & Trainer
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "BENIGN", 1: "MALICIOUS"},
    label2id={"BENIGN": 0, "MALICIOUS": 1}
)

training_args = TrainingArguments(
    output_dir="./distilbert_checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=25,
    save_total_limit=1,
    report_to="none"
)

trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Starting DistilBERT Fine-Tuning...")
trainer.train()

In [ ]:
# Step 8: Comprehensive Test Set Evaluation & Confusion Matrix
print("Evaluating on Unseen Test Dataset...")
test_predictions = trainer.predict(test_dataset)
preds = np.argmax(test_predictions.predictions, axis=-1)
labels = test_predictions.label_ids

tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
cm = np.array([[tn, fp], [fn, tp]])

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["BENIGN", "MALICIOUS"], yticklabels=["BENIGN", "MALICIOUS"])
plt.title("DistilBERT WAF - Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=300)
plt.show()

print("Classification Report:")
print(classification_report(labels, preds, target_names=["BENIGN", "MALICIOUS"]))
print(f"False Positive Rate (FPR): {fp / (fp + tn):.4%}")
print(f"False Negative Rate (FNR): {fn / (fn + tp):.4%}")

In [ ]:
# Step 9: Save Model and Download Zip for Local Deployment
output_dir = "./distilbert_waf"
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

!zip -r distilbert_waf.zip ./distilbert_waf
from google.colab import files
files.download("distilbert_waf.zip")
print("Model weights and tokenizer exported successfully!")